<a href="https://colab.research.google.com/github/dramaqueenvee/RLHF-lexical-overuse-in-Spanish-a-pilot-study/blob/main/gemma2_instruct_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

GEMMA INSTRUCT MODEL

In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 45.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
# Load instruct model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

print("Instruct model loaded successfully!")

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Instruct model loaded successfully!


In [ ]:
import pandas as pd
import torch
import time

df_all = pd.read_csv("spanish_abstracts.csv")
df_base = pd.read_csv("base_continuations_clean.csv")
df_filtered = df_all[df_all["pmid"].isin(df_base["pmid"])].reset_index(drop=True)
print(f"Running instruct on {len(df_filtered)} abstracts")

instruct_continuations = []
start_total = time.time()

for i, row in df_filtered.iterrows():
    messages = [
        {
            "role": "user",
            "content": f'Continúa el siguiente texto científico escribiendo solo la continuación, sin comentarios ni preguntas: "{row["first_half"]}"'
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            repetition_penalty=1.3,
            no_repeat_ngram_size=4
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    continuation = generated.split(row["first_half"])[-1].strip()

    instruct_continuations.append({
        "pmid": row["pmid"],
        "continuation": continuation
    })

    elapsed = time.time() - start_total
    avg = elapsed / (i + 1)
    remaining = avg * (len(df_filtered) - i - 1)
    print(f"Processed {i+1}/{len(df_filtered)} — ~{int(remaining/60)} min remaining")

instruct_df = pd.DataFrame(instruct_continuations)
instruct_df.to_csv("instruct_gemma2_continuations.csv", index=False, encoding="utf-8-sig")
print(f"Done! Saved {len(instruct_df)} continuations")
print("\nExample")
print(instruct_df.iloc[2]["continuation"])

Running instruct on 49 abstracts
Processed 1/49 — ~5 min remaining
Processed 2/49 — ~3 min remaining
Processed 3/49 — ~3 min remaining
Processed 4/49 — ~2 min remaining
Processed 5/49 — ~2 min remaining
Processed 6/49 — ~2 min remaining
Processed 7/49 — ~1 min remaining
Processed 8/49 — ~1 min remaining
Processed 9/49 — ~1 min remaining
Processed 10/49 — ~1 min remaining
Processed 11/49 — ~1 min remaining
Processed 12/49 — ~1 min remaining
Processed 13/49 — ~1 min remaining
Processed 14/49 — ~1 min remaining
Processed 15/49 — ~1 min remaining
Processed 16/49 — ~1 min remaining
Processed 17/49 — ~1 min remaining
Processed 18/49 — ~1 min remaining
Processed 19/49 — ~1 min remaining
Processed 20/49 — ~1 min remaining
Processed 21/49 — ~1 min remaining
Processed 22/49 — ~1 min remaining
Processed 23/49 — ~1 min remaining
Processed 24/49 — ~1 min remaining
Processed 25/49 — ~1 min remaining
Processed 26/49 — ~0 min remaining
Processed 27/49 — ~0 min remaining
Processed 28/49 — ~0 min remain

In [ ]:
import pandas as pd
from collections import Counter
from scipy.stats import chi2_contingency
import numpy as np
import re

base_df = pd.read_csv("base_continuations_clean.csv")
instruct_df = pd.read_csv("instruct_gemma2_continuations.csv")

print(f"Base continuations: {len(base_df)}")
print(f"Instruct continuations: {len(instruct_df)}")

def get_word_counts(df):
    all_text = " ".join(df["continuation"].dropna().tolist()).lower()
    words = re.findall(r'\b[a-záéíóúüñ]+\b', all_text)
    return Counter(words)

base_counts = get_word_counts(base_df)
instruct_counts = get_word_counts(instruct_df)

base_total = sum(base_counts.values())
instruct_total = sum(instruct_counts.values())

print(f"Base total words: {base_total}")
print(f"Instruct total words: {instruct_total}")

stopwords = {
    "de", "la", "el", "los", "las", "y", "en", "se", "que", "un", "una",
    "con", "por", "del", "al", "es", "su", "sus", "para", "como", "más",
    "este", "esta", "estos", "estas", "entre", "sin", "sobre", "o", "a",
    "no", "lo", "le", "les", "pero", "fue", "son", "hay", "has", "han",
    "ser", "estar", "siendo", "sido", "era", "eran", "también", "así"
}

stopwords.add("continuación")
stopwords.add("texto")
stopwords.add("científico")
stopwords.add("siguiente")
stopwords.add("resumen")

results = []
all_words = set(instruct_counts.keys()) | set(base_counts.keys())

for word in all_words:
    if word in stopwords or len(word) <= 3 or word.isupper():
        continue

    b = base_counts.get(word, 0)
    i = instruct_counts.get(word, 0)

    if (b + i) < 5:
        continue

    observed = np.array([[i, instruct_total - i],
                         [b, base_total - b]])
    try:
        chi2, p, _, _ = chi2_contingency(observed)
    except:
        continue

    instruct_opm = (i / instruct_total) * 1e6
    base_opm = (b / base_total) * 1e6

    if instruct_opm > base_opm:
        results.append((word, round(base_opm), round(instruct_opm), round(p, 4)))

results.sort(key=lambda x: x[3])

print("\nContent words overused in instruct (p < 0.15):")
print(f"{'Word':<25} {'Base opm':<12} {'Instruct opm':<15} {'p-value'}")
print("-" * 65)
for word, b, i, p in results[:30]:
    if p < 0.15:
        print(f"{word:<25} {b:<12} {i:<15} {p}")

Base continuations: 49
Instruct continuations: 49
Base total words: 2344
Instruct total words: 1362

Content words overused in instruct (p < 0.15):
Word                      Base opm     Instruct opm    p-value
-----------------------------------------------------------------
contexto                  427          3671            0.0518
tratamiento               427          3671            0.0518
médica                    427          2937            0.1228
salud                     427          2937            0.1228
pacientes                 1280         4405            0.1291
